In [ ]:
import os
import subprocess
import re
from datetime import datetime
from random import shuffle

# -------------------------------------------------------------------
# 1. ОСНОВНЫЕ НАСТРОЙКИ (Изменяйте при необходимости)
# -------------------------------------------------------------------
WORK_DIR = "Olimp_2026"
validation_split = 0.2
classes = ["vehicle"]
RESUME_TRAINING = True  # Флаг продолжения обучения с последних весов

# -------------------------------------------------------------------
# 2. ПРОВЕРКА ОКРУЖЕНИЯ И МОНТИРОВАНИЕ ДИСКА
# -------------------------------------------------------------------
if not os.path.exists('/mydrive'):
    from google.colab import drive
    drive.mount('/content/gdrive')
    !ln -s /content/gdrive/MyDrive/ /mydrive
else:
    print("Google Диск уже смонтирован.")

# -------------------------------------------------------------------
# 3. СБОРКА DARKNET (Выполняется только 1 раз)
# -------------------------------------------------------------------
if not os.path.exists('/content/darknet'):
    !git clone https://github.com/AlexeyAB/darknet /content/darknet
    os.chdir('/content/darknet')
    !sed -i 's/GPU=0/GPU=1/' Makefile
    !sed -i 's/CUDNN=0/CUDNN=1/' Makefile
    !sed -i 's/CUDNN_HALF=0/CUDNN_HALF=1/' Makefile
    !make
else:
    print("Репозиторий Darknet уже скомпилирован. Пропускаем сборку.")

# Переходим в рабочую папку Darknet по абсолютному пути
os.chdir('/content/darknet')

# -------------------------------------------------------------------
# 4. РАСПАКОВКА ДАТАСЕТА (Выполняется только 1 раз)
# -------------------------------------------------------------------
if not os.path.exists('/content/darknet/data/obj'):
    os.system(f'cp /mydrive/{WORK_DIR}/obj.zip /content/')
    os.system('unzip /content/obj.zip -d /content/darknet/data/')
else:
    print("Датасет уже распакован. Пропускаем unzip.")

# -------------------------------------------------------------------
# 5. СОЗДАНИЕ ПАПКИ БЭКАПА С МЕТКОЙ ВРЕМЕНИ
# -------------------------------------------------------------------
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
backup_path = f'/mydrive/{WORK_DIR}/backup_{timestamp}'
os.makedirs(backup_path, exist_ok=True)
print(f"Папка для сохранения бэкапа этой сессии: {backup_path}")

# -------------------------------------------------------------------
# 6. ГЕНЕРАЦИЯ TRAIN/TEST СПИСКОВ И OBJ.NAMES
# -------------------------------------------------------------------
files = os.listdir('data/obj/')
images = ["data/obj/" + file for file in files if file.endswith('.jpg')]
shuffle(images)

train_len = int((1 - validation_split) * len(images))
train_dataset = images[:train_len]
valid_dataset = images[train_len:]

with open('data/train.txt', 'w') as f:
    f.write('\n'.join(train_dataset))

with open('data/test.txt', 'w') as f:
    if valid_dataset:
        f.write('\n'.join(valid_dataset))

classes_amount = len(classes)
with open('data/obj.names', 'w') as f:
    f.write('\n'.join(classes))

# Сохраняем obj.names в папку бэкапа
os.system(f'cp data/obj.names {backup_path}/')

with open('data/obj.data', 'w') as f:
    f.write(f"classes={classes_amount}\ntrain=data/train.txt\nvalid=data/test.txt\nnames=data/obj.names\nbackup={backup_path}")

# -------------------------------------------------------------------
# 7. ПОДГОТОВКА И НАСТРОЙКА CONFIG ФАЙЛА
# -------------------------------------------------------------------
template_file = "cfg/yolov4-tiny-custom.cfg"
config_file = "cfg/yolov4-tiny-obj.cfg"
max_batches = max(classes_amount * 2000, 6000)
step1 = int(max_batches * 0.8)
step2 = int(max_batches * 0.9)
filters = (classes_amount + 5) * 3

os.system(f'cp {template_file} -d {config_file}')
os.system(f"sed -i 's/steps=400000,450000/steps={step1},{step2}/' {config_file}")
os.system(f"sed -i 's/classes=80/classes={classes_amount}/' {config_file}")
os.system(f"sed -i 's/filters=255/filters={filters}/' {config_file}")
os.system(f"sed -i 's/max_batches = 500200/max_batches = {max_batches}/' {config_file}")

# -------------------------------------------------------------------
# 8. РАСЧЁТ ANCHORS И ОПТИМИЗАЦИЯ CONFIG
# -------------------------------------------------------------------
print("Расчет anchors...")
result = subprocess.run(
    ["./darknet", "detector", "calc_anchors", "data/obj.data", "-num_of_clusters", "6", "-width", "416", "-height", "416"],
    capture_output=True, text=True
)

anchors_match = re.search(r'anchors\s*=\s*[\d,\s]+', result.stdout)
if anchors_match:
    new_anchors = anchors_match.group(0).strip()
    print(f"Найдены новые якоря: {new_anchors}")
    os.system(f"sed -i 's/^anchors\s*=.*/{new_anchors}/' {config_file}")
else:
    print("Предупреждение: не удалось автоматически расчитать anchors.")

# Копируем итоговый .cfg файл в папку бэкапа
os.system(f'cp {config_file} {backup_path}/')

# -------------------------------------------------------------------
# 9. ВЫБОР ВЕСОВ И ЗАПУСК ОБУЧЕНИЯ
# -------------------------------------------------------------------
weights_to_use = "yolov4-tiny.conv.29"

if RESUME_TRAINING:
    mydrive_path = f"/mydrive/{WORK_DIR}"
    if os.path.exists(mydrive_path):
        # Ищем все папки бэкапов, созданные ранее
        backup_dirs = [os.path.join(mydrive_path, d) for d in os.listdir(mydrive_path) 
                       if os.path.isdir(os.path.join(mydrive_path, d)) and d.startswith('backup_')]
        
        if backup_dirs:
            # Находим самую последнюю папку по времени создания
            latest_backup_dir = max(backup_dirs, key=os.path.getmtime)
            print(لت"Обнаружена последняя папка с бэкапами: {latest_backup_dir}")
            
            # Ищем веса по заданному приоритету: last -> с номером итерации -> best
            last_weights = [os.path.join(latest_backup_dir, f) for f in os.listdir(latest_backup_dir) if f.endswith('_last.weights')]
            iter_weights = [os.path.join(latest_backup_dir, f) for f in os.listdir(latest_backup_dir) if re.search(r'_\d+\.weights$', f)]
            best_weights = [os.path.join(latest_backup_dir, f) for f in os.listdir(latest_backup_dir) if f.endswith('_best.weights')]
            
            if last_weights:
                weights_to_use = last_weights[0]
            elif iter_weights:
                # Сортируем по номеру итерации, если их несколько
                iter_weights.sort(key=lambda x: int(re.search(r'_(\d+)\.weights$', x).group(1)))
                weights_to_use = iter_weights[-1]
            elif best_weights:
                weights_to_use = best_weights[0]
                
            print(f"Выбраны веса для продолжения обучения: {weights_to_use}")
        else:
            print("Предыдущие папки бэкапов не найдены. Используем базовые веса yolov4-tiny.conv.29")
    else:
        print("Рабочая директория на диске не найдена. Используем базовые веса yolov4-tiny.conv.29")

# Скачиваем базовые веса на случай, если они нужны и не указаны другие
if weights_to_use == "yolov4-tiny.conv.29" and not os.path.exists('yolov4-tiny.conv.29'):
    !wget https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v4_pre/yolov4-tiny.conv.29

print("Запуск обучения...")
!./darknet detector train data/obj.data cfg/yolov4-tiny-obj.cfg {weights_to_use} -dont_show -map
